# 🌿 Semana 11 · Laboratorio: Archivos y Pandas

**Del CSV al análisis de datos ecológicos**

---
Archivo → Guardar una copia en Drive. Completa `# TU CÓDIGO AQUÍ`.

## Cargar datos

In [ ]:
import pandas as pd
import math
from google.colab import drive

drive.mount('/content/drive')

# Ajusta la ruta a tu carpeta en Drive
ruta_bosque = "/content/drive/MyDrive/CBIT010_Archivos_ayudantia/Semana11/bosque_valdiviano_simple.csv"
df = pd.read_csv(ruta_bosque)
print(f"✅ {df.shape[0]} filas, {df.shape[1]} columnas cargadas")

## Funciones de diversidad (de la Sem. 10 — copiar aquí para reutilizar)

In [ ]:
def shannon_entropy(abundancias):
    """H' en nats."""
    total = sum(abundancias)
    H = 0
    for n in abundancias:
        if n > 0:
            p = n / total
            H -= p * math.log(p)
    return H

def equitatividad(abundancias):
    """J = H' / ln(S)."""
    S = len(abundancias)
    if S <= 1: return 0.0
    return shannon_entropy(abundancias) / math.log(S)

def simpson_index(abundancias):
    """1 - D."""
    total = sum(abundancias)
    D = sum((n / total) ** 2 for n in abundancias)
    return 1 - D

---
## Ejercicio 1: Exploración del dataset 🔍

In [ ]:
# 1a. Dimensiones y tipos
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nTipos de datos:")
print(df.dtypes)

In [ ]:
# 1b. Primeras y últimas filas
df.head()

In [ ]:
# 1c. Estadísticas descriptivas
df.describe()

In [ ]:
# 1d. Responde:
# ¿Cuántas especies diferentes hay?
n_especies = ___   # df["especie"].nunique()
print(f"Número de especies: {n_especies}")

# ¿Cuál es el DAP promedio?
dap_promedio = ___  # df["dap_cm"].mean()
print(f"DAP promedio: {dap_promedio:.1f} cm")

# ¿Cuántos árboles por parcela?
print("\nÁrboles por parcela:")
print(___)  # df["parcela"].value_counts().sort_index()

---
## Ejercicio 2: Filtrado 🎯

In [ ]:
# 2a. Árboles con DAP > 40 cm
grandes = df[___]
print(f"Árboles con DAP > 40 cm: {len(grandes)} de {len(df)}")
grandes.head()

In [ ]:
# 2b. Solo Nothofagus dombeyi
coigues = df[df["especie"] == ___]
print(f"Coigüe: {len(coigues)} árboles")
print(f"DAP promedio coigüe: {coigues['dap_cm'].mean():.1f} cm")

In [ ]:
# 2c. Parcela 3 con altura > 15 m
filtro = df[(df["parcela"] == ___) & (df["altura_m"] > ___)]
print(f"Parcela 3, altura > 15m: {len(filtro)} árboles")

In [ ]:
# 2d. ¿Cuál es el árbol más alto del dataset?
idx_max = df["altura_m"].idxmax()
arbol_alto = df.loc[idx_max]
print(f"Árbol más alto:\n{arbol_alto}")

---
## Ejercicio 3: Conteos y frecuencias 📊

In [ ]:
# 3a. Tabla de frecuencias de especies
print("Abundancia por especie:")
print(df["especie"].value_counts())

In [ ]:
# 3b. Tabla cruzada: especies × parcelas
tabla = pd.crosstab(df["parcela"], df["especie"])
print(tabla)

In [ ]:
# 3c. ¿Cuál parcela tiene más especies únicas?
sp_por_parcela = df.groupby("parcela")["especie"].nunique()
print("\nEspecies únicas por parcela:")
print(sp_por_parcela)
print(f"\nParcela con más especies: {sp_por_parcela.idxmax()}")

---
## Ejercicio 4: groupby — agrupar y resumir 📈

In [ ]:
# 4a. DAP promedio por especie
print("DAP promedio por especie:")
print(df.groupby("especie")["dap_cm"].mean().round(1))

In [ ]:
# 4b. Tabla completa por especie
resumen_sp = df.groupby("especie")["dap_cm"].agg(["count", "mean", "std", "min", "max"])
resumen_sp = resumen_sp.round(1)
print(resumen_sp)

In [ ]:
# 4c. Estadísticas por parcela
# TU CÓDIGO AQUÍ — crea un resumen con mean y std de dap_cm y altura_m por parcela
resumen_parcela = df.groupby("parcela").agg({
    "dap_cm": [___],      # "mean", "std"
    "altura_m": [___],    # "mean", "std"
    "especie": "nunique"
})
print(resumen_parcela)

In [ ]:
# 4d. ¿Cuál especie tiene el mayor DAP promedio?
dap_por_sp = df.groupby("especie")["dap_cm"].mean()
especie_max = dap_por_sp.idxmax()
print(f"Especie con mayor DAP promedio: {especie_max} ({dap_por_sp[especie_max]:.1f} cm)")

---
## Ejercicio 5: Aplicar funciones propias 🔧

In [ ]:
# 5a. Shannon H' del dataset completo
conteos_total = df["especie"].value_counts()
H_total = shannon_entropy(conteos_total.values)
J_total = equitatividad(conteos_total.values)
print(f"H' total: {H_total:.4f} nats")
print(f"J total: {J_total:.4f}")

In [ ]:
# 5b. Shannon H' por parcela
print("\nDiversidad por parcela:")
print(f"{'Parcela':>8} {'S':>4} {'N':>5} {'H_nats':>8} {'J':>6}")
print("-" * 35)

for parcela, grupo in df.groupby("parcela"):
    conteos = grupo["especie"].value_counts()
    H = shannon_entropy(conteos.values)
    J = equitatividad(conteos.values)
    S = len(conteos)
    N = len(grupo)
    print(f"{parcela:>8} {S:>4} {N:>5} {H:>8.4f} {J:>6.3f}")

In [ ]:
# 5c. Crear columna 'tamano' con .apply()
def clasificar_tamano(dap):
    if dap > 40:
        return "grande"
    elif dap > 20:
        return "mediano"
    else:
        return "pequeño"

df["tamano"] = df["dap_cm"].apply(___)   # clasificar_tamano
print(df["tamano"].value_counts())

In [ ]:
# 5d. Conteo de tamaños por parcela
print("\nTamaño por parcela:")
print(pd.crosstab(df["parcela"], df["tamano"]))

---
## 🏆 Ejercicio 6 (Desafío): Dataset de cámaras trampa

In [ ]:
# Cargar segundo dataset
ruta_camaras = "/content/drive/MyDrive/CBIT010_Archivos_ayudantia/Semana11/camaras_trampa_simulado.csv"
cam = pd.read_csv(ruta_camaras)
print(f"✅ {cam.shape[0]} registros cargados")
cam.head()

In [ ]:
# 6a. Exploración básica
print(f"Sitios: {cam['sitio'].nunique()}")
print(f"Especies: {cam['especie'].nunique()}")
print(f"\nRegistros por sitio:")
print(cam["sitio"].value_counts())

In [ ]:
# 6b. Tabla cruzada sitio × especie
tabla_cam = pd.crosstab(cam["sitio"], cam["especie"])
print(tabla_cam)

In [ ]:
# 6c. Shannon H' por sitio
# TU CÓDIGO AQUÍ
print("\nDiversidad por sitio:")
for sitio, grupo in cam.groupby("sitio"):
    conteos = grupo["especie"].value_counts()
    H = shannon_entropy(conteos.values)
    J = equitatividad(conteos.values)
    S = len(conteos)
    print(f"  {sitio}: S={S}, H'={H:.4f}, J={J:.3f}")

In [ ]:
# 6d. ¿Hay alguna especie que falte en algún sitio?
# Pista: buscar ceros en la tabla cruzada
for sitio in tabla_cam.index:
    ausentes = tabla_cam.columns[tabla_cam.loc[sitio] == 0].tolist()
    if ausentes:
        print(f"{sitio}: falta(n) {ausentes}")
    else:
        print(f"{sitio}: todas las especies presentes")

**Reflexión:** ¿El sitio con más registros es el más diverso? ¿O la diversidad depende más de la equitatividad que del número de observaciones?

---
## 🏁 Fin del laboratorio

**Checklist:**
- [ ] ¿`df.describe()` funciona sobre el dataset forestal?
- [ ] ¿Los filtrados del Ej. 2 dan resultados razonables?
- [ ] ¿`groupby` produce tablas por especie y parcela?
- [ ] ¿H' por parcela da valores distintos?
- [ ] (Desafío) ¿Identificaron la especie ausente en algún sitio?

---

## 📝 Tarea evaluada #2
Se publica en Notion. **Entrega:** Semana 12. Datos: `camaras_trampa_simulado.csv`.

---
*Semana 11 · Archivos y Pandas · UACh 2026*